# Урок 7: чанкинг и эмбеддинги

Проект 2 — RAG-ассистент «спроси у документа». Ноутбук выполняется в Google Colab (первая ячейка ставит зависимости); здесь сохранена финальная версия кода вместе с результатами прогона на тестовых документах из `data/sample_docs/`.

In [ ]:
!pip install -q sentence-transformers

## Модель

`BAAI/bge-m3` — многоязычная модель эмбеддингов (100+ языков, включая русский и украинский), вектор размерности 1024. Выбрана вместо TF.js Universal Sentence Encoder, у которого нет поддерживаемой RU/UA-версии, и вместо `deepvk/USER-bge-m3`, заточенной только под русский.

In [ ]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("BAAI/bge-m3")

## Чанкинг

Скользящее окно по словам с нахлёстом (overlap), чтобы не обрывать мысль ровно на границе чанка.

In [ ]:
def chunk_text(text: str, chunk_size: int = 150, overlap: int = 30) -> list[str]:
    words = text.split()
    chunks = []
    start = 0
    while start < len(words):
        end = start + chunk_size
        chunks.append(" ".join(words[start:end]))
        start += chunk_size - overlap
    return chunks

## Тестовые документы

Два варианта одной и той же политики возврата/доставки — на русском и на украинском (`data/sample_docs/policy_ru.txt`, `policy_ua.txt`).

In [ ]:
with open("data/sample_docs/policy_ru.txt", encoding="utf-8") as f:
    document_ru = f.read()

with open("data/sample_docs/policy_ua.txt", encoding="utf-8") as f:
    document_ua = f.read()

In [ ]:
chunks_ru = chunk_text(document_ru)
chunks_ua = chunk_text(document_ua)

# объединяем чанки обоих документов в один список — эмбеддинги считаем разом,
# но не теряем, из какого документа какой чанк пришёл (пригодится в Уроке 8)
chunks = (
    [{"text": c, "source": "policy_ru.txt"} for c in chunks_ru]
    + [{"text": c, "source": "policy_ua.txt"} for c in chunks_ua]
)

embeddings = model.encode([c["text"] for c in chunks], normalize_embeddings=True)

print(f"чанков: {len(chunks)}, форма эмбеддингов: {embeddings.shape}")

## Проверка на вшивость — кросс-языковое сходство

Если модель действительно понимает смысл, а не просто буквы, то русская и украинская версии одного и того же вопроса должны получить высокое сходство, а не связанная по теме русская фраза — низкое.

In [ ]:
sentences = [
    "Как оформить возврат бракованного товара?",       # RU
    "Як оформити повернення бракованого товару?",       # UA, тот же смысл
    "Обожаю готовить пасту по выходным.",                # RU, не по теме
]

vecs = model.encode(sentences, normalize_embeddings=True)

print("RU-UA (должно быть высоким):", vecs[0] @ vecs[1])
print("RU-паста (должно быть низким):", vecs[0] @ vecs[2])

**Результат прогона:**

- RU-UA: `0.9793576`
- RU-паста: `0.34002185`

Разница почти в три раза — модель уверенно отличает смысловую близость от простого совпадения языка/слов. Проверка пройдена, можно двигаться в Урок 8 (векторный поиск).

In [ ]:
import numpy as np

def search(query: str, chunks: list[dict], embeddings: np.ndarray, k: int = 3):
    query_vec = model.encode([query], normalize_embeddings=True)[0]
    scores = embeddings @ query_vec               # сходство со всеми чанками разом
    top_indices = np.argsort(scores)[::-1][:k]     # индексы k наибольших значений

    return [
        {**chunks[i], "score": float(scores[i])}
        for i in top_indices
    ]

In [ ]:
for r in search("Скільки часу займає доставка по країні?", chunks, embeddings, k=3):
    print(f"[{r['score']:.3f}] ({r['source']}) {r['text'][:180]}...")

In [ ]:
test_queries = [
    {"query": "Сколько дней есть на возврат товара?", "expected_source": "policy_ru.txt"},
    {"query": "Скільки часу займає доставка по країні?", "expected_source": "policy_ua.txt"},
    {"query": "Что делать при повреждённом товаре?", "expected_source": "policy_ru.txt"},
    # добавь ещё 3-5 своих вопросов
]

hits = 0
for tq in test_queries:
    results = search(tq["query"], chunks, embeddings, k=3)
    found = any(r["source"] == tq["expected_source"] for r in results)
    hits += found
    print(f"{'✓' if found else '✗'} {tq['query']}")

print(f"\nHit rate: {hits}/{len(test_queries)}")